In [ ]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
url = "https://search.naver.com/search.naver"
my_params = {"where":"news","query":"인공지능"}
response = requests.get(url,params=my_params,timeout=10)
# print("상태코드",response.status_code)
# print("내용",response.text)
soup = BeautifulSoup(response.text, "html.parser")
# print(soup.prettify())
news_list = soup.select(".fds-news-item-list-tab > div")
# print(len(news_list))
data=[]
for item in news_list:
    title = item.select_one('a[data-heatmap-target=".tit"]')
    desc = item.select_one('a[data-heatmap-target=".body"]')
    press = item.select_one(".sds-comps-profile-info-title")
    url = title.get("href")
    # print(title.get_text(" ",strip=True).replace("새 창 열림","").strip())
    # print(desc.get_text(" ",strip=True).strip())
    # print(press.get_text(" ",strip=True).strip().replace("새 창 열림","").strip())
        # print("==="*30)
    strip_title =  title.get_text(" ",strip=True).replace("새 창 열림","").strip()
    strip_press= press.get_text(" ",strip=True).strip().replace("새 창 열림","").strip()
    strip_desc = desc.get_text(" ",strip=True).strip()


    data.append({"제목":strip_title,"언론사":strip_press,"내용":strip_desc,"URL":url})
df = pd.DataFrame(data)
df.to_excel("네이버뉴스.xlsx",index=False,engine="xlsxwriter")




        


In [ ]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
url = "https://search.daum.net/search"
my_params = {"w":"news","q":"인공지능"}
response = requests.get(url,params=my_params,timeout=10)
# print("상태코드",response.status_code)
# print("내용",response.text)
soup = BeautifulSoup(response.text, "html.parser")
# print(soup.prettify())
news_list = soup.select(".c-list-basic > li")
# print(len(news_list))
data=[]
for item in news_list:
    title = item.select_one(".item-title a")
    strip_title=title.get_text(" ",strip=True)
    desc = item.select_one(".item-contents")
    strip_desc= desc.get_text(" ",strip=True)
    press= item.select_one(".c-tit-doc .txt_info")
    strip_press= press.get_text(" ",strip=True)

    url = title.get("href")

    data.append({"제목":strip_title,"언론사":strip_press,"내용":desc,"URL":url})
df = pd.DataFrame(data)
df.to_excel("다음뉴스.xlsx",index=False,engine="xlsxwriter")




        


In [39]:
# pip install xlsxwriter
# pip install requests
# pip install beautifulsoup4
# pip install pandas


import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
url = "https://search.daum.net/search"
data=[]
for page in range(1,11):
    print(page)
    my_params = {"w":"news","q":"인공지능","p":page}
    response = requests.get(url,params=my_params,timeout=10)
    # print("상태코드",response.status_code)
    # print("내용",response.text)
    soup = BeautifulSoup(response.text, "html.parser")
    # print(soup.prettify())
    news_list = soup.select(".c-list-basic > li")
    # print(len(news_list))
    
    for item in news_list:
        title = item.select_one(".item-title a")
        strip_title=title.get_text(" ",strip=True)
        desc = item.select_one(".item-contents")
        strip_desc= desc.get_text(" ",strip=True)
        press= item.select_one(".c-tit-doc .txt_info")
        strip_press= press.get_text(" ",strip=True)

        news_url = title.get("href")

        data.append({"제목":strip_title,"언론사":strip_press,"내용":strip_desc,"URL":news_url})
    time.sleep(1)
df = pd.DataFrame(data)
df.to_excel("다음뉴스.xlsx",index=False,engine="xlsxwriter")




        


1
2
3
4
5
6
7
8
9
10


In [41]:
# pip install selenium
from selenium import webdriver
driver = webdriver.Chrome()



In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()

# 검색 결과 페이지로 바로 이동
url = (
    "https://search.naver.com/search.naver?where=news&ie=utf8&sm=nws_hty&query=트럼프 중간선거"
)

driver.get(url)

wait = WebDriverWait(driver, 10)

# 뉴스 목록 로딩 대기
wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, ".fds-news-item-list-tab > div")
    )
)

data = []
collected = set()

# 스크롤 30회
for i in range(10):

    news_list = driver.find_elements(
        By.CSS_SELECTOR,
        ".fds-news-item-list-tab > div"
    )

    for news in news_list:

        titles = news.find_elements(
            By.CSS_SELECTOR,
            'a[data-heatmap-target=".tit"]'
        )

        if not titles:
            continue

        title = titles[0].text
        link = titles[0].get_attribute("href")

        if not link or link in collected:
            continue

        descs = news.find_elements(
            By.CSS_SELECTOR,
            'a[data-heatmap-target=".body"]'
        )

        presses = news.find_elements(
            By.CSS_SELECTOR,
            ".sds-comps-profile-info-title-text"
        )

        desc = descs[0].text if descs else ""
        press = presses[0].text if presses else ""

        data.append({
            "제목": title,
            "내용": desc,
            "언론사": press,
            "링크": link
        })

        collected.add(link)

    print(f"{i+1}번째 스크롤 / 누적 {len(data)}개")

    driver.execute_script(
        "window.scrollBy(0, 800);"
    )

    time.sleep(2)

# 엑셀 저장
df = pd.DataFrame(data)
df.to_excel("네이버뉴스.xlsx", index=False)

print("최종 수집:", len(df), "개")

driver.quit()

1번째 스크롤 / 누적 10개
2번째 스크롤 / 누적 10개
3번째 스크롤 / 누적 19개
4번째 스크롤 / 누적 19개
5번째 스크롤 / 누적 29개
6번째 스크롤 / 누적 29개
7번째 스크롤 / 누적 29개
8번째 스크롤 / 누적 39개
9번째 스크롤 / 누적 39개
10번째 스크롤 / 누적 49개
최종 수집: 49 개


In [11]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

driver = webdriver.Chrome()

url = "https://www.coupang.com/np/search?q=사과"
driver.get(url)

wait = WebDriverWait(driver, 15)

# 상품 목록 로딩 대기
wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "#product-list > li")
    )
)

# 상품 목록
items = driver.find_elements(
    By.CSS_SELECTOR, "#product-list > li"
)

print("상품 개수:", len(items))

data = []

for item in items:

    # 상품명
    titles = item.find_elements(
        By.CSS_SELECTOR,
        '[class*="ProductUnit_productNameV2"]'
    )

    if not titles:
        continue

    title = titles[0].text

    # 판매가격
    prices = item.find_elements(
        By.CSS_SELECTOR,
        '[class*="PriceArea_priceArea"] '
        'div[class*="fw-text-[20px]"] span'
    )

    price = prices[0].text if prices else ""

    # 원래가격
    originals = item.find_elements(
        By.CSS_SELECTOR,
        '[class*="PriceArea_priceArea"] del'
    )

    original = originals[0].text if originals else ""

    # 상품 링크
    links = item.find_elements(
        By.CSS_SELECTOR,
        'a[href*="/vp/products/"]'
    )

    link = links[0].get_attribute("href") if links else ""

    # 평점
    ratings = item.find_elements(
        By.CSS_SELECTOR,
        '[class*="ProductRating_productRating"] [aria-label]'
    )

    rating = (
        ratings[0].get_attribute("aria-label")
        if ratings else ""
    )

    print(title, price, rating)

    data.append({
        "상품명": title,
        "판매가격": price,
        "원래가격": original,
        "평점": rating,
        "링크": link
    })

# 엑셀 저장
df = pd.DataFrame(data)
df.to_excel("쿠팡_사과.xlsx", index=False)

print("수집 완료:", len(df), "개")

driver.quit()

InvalidSessionIdException: Message: invalid session id: session deleted as the browser has closed the connection
from disconnected: not connected to DevTools
  (Session info: chrome=156.0.8078.80); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	chromedriver!GetHandleVerifier [0x7ff6aa46be65+4135]
	chromedriver!(No symbol) [0x7ff6aa3735f0]
	chromedriver!(No symbol) [0x7ff6aa1e26d3]
	chromedriver!(No symbol) [0x7ff6aa1d7b2f]
	chromedriver!(No symbol) [0x7ff6aa1e8859]
	chromedriver!(No symbol) [0x7ff6aa211fe6]
	chromedriver!(No symbol) [0x7ff6aa21639f]
	chromedriver!(No symbol) [0x7ff6aa1fb969]
	chromedriver!(No symbol) [0x7ff6aa1fcca3]
	chromedriver!GetHandleVerifier [0x7ff6aa8e5af1+47ddc1]
	chromedriver!GetHandleVerifier [0x7ff6aa9067f7+49eac7]
	chromedriver!GetHandleVerifier [0x7ff6aa8fce20+4950f0]
	chromedriver!GetHandleVerifier [0x7ff6aa58ff8e+12825e]
	chromedriver!(No symbo

In [14]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

url = "https://apis.data.go.kr/B552481/StockSvc/getStkListInfoN1"

params = {
    "serviceKey": "Wnus4QpirWGI56CfvzMWDIDHMRL/mEF/qTl9gwVNbRggLYTGPFIdwBy0L51B+27d5QRbLanNmIAxPwNvl7dKPA==",
    "numOfRows": 100,
    "pageNo": 1,
    "isin": "KR7005930003"
}

response = requests.get(url, params=params, timeout=20)

print("HTTP 상태코드:", response.status_code)
print("요청 URL:", response.url)
print("응답 내용:", response.text[:1000])

response.raise_for_status()

# XML 데이터 추출
root = ET.fromstring(response.content)

data = []

for item in root.findall(".//item"):
    row = {
        child.tag: child.text
        for child in item
    }
    data.append(row)

df = pd.DataFrame(data)

print(df)

if not df.empty:
    df.to_excel("삼성전자_주식정보.xlsx", index=False)
    print("엑셀 저장 완료")
else:
    print("조회된 데이터가 없습니다.")

HTTP 상태코드: 200
요청 URL: https://apis.data.go.kr/B552481/StockSvc/getStkListInfoN1?serviceKey=Wnus4QpirWGI56CfvzMWDIDHMRL%2FmEF%2FqTl9gwVNbRggLYTGPFIdwBy0L51B%2B27d5QRbLanNmIAxPwNvl7dKPA%3D%3D&numOfRows=100&pageNo=1&isin=KR7005930003
응답 내용: <response>
    <header>
        <resultCode>00</resultCode>
        <resultMsg>NORMAL_SERVICE</resultMsg>
    </header>
    <body>
        <items>
            <item>
                <isin>KR7005930003</isin>
                <korSecnNm>삼성전자</korSecnNm>
                <issucoCustno>593</issucoCustno>
                <listTpcd>110</listTpcd>
                <apliDt>19750611</apliDt>
                <xpitDt>99991231</xpitDt>
                <dlistDt>99991231</dlistDt>
            </item>
            <item>
                <isin>KR7005930003</isin>
                <korSecnNm>삼성전자</korSecnNm>
                <issucoCustno>593</issucoCustno>
                <listTpcd>211</listTpcd>
                <apliDt>19750611</apliDt>
                <xpitDt>20190912</

In [3]:
import folium

# 서울시청 중심 지도
m = folium.Map(
    location=[37.5665, 126.9780],
    zoom_start=12
)
m.save("map.html")
